In [6]:
import pandas as pd
import numpy as np
import json
import google.generativeai as genai
from google.api_core import retry
import time
from typing import Dict, Any
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, mean_absolute_error
from dotenv import load_dotenv
import os

In [7]:
load_dotenv()
np.random.seed(42) # random seed for reproducibility

## 1. Data Loading and Sampling

In [8]:
df = pd.read_csv(f'D:\Work\Fynd\yelp.csv')
print(df.shape)
print(f'Columns:',df.columns.tolist())
print(f"Stars:\n", df['stars'].value_counts().sort_index()) # displays count of each star rating with a sorted index


(10000, 10)
Columns: ['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id', 'cool', 'useful', 'funny']
Stars:
 stars
1     749
2     927
3    1461
4    3526
5    3337
Name: count, dtype: int64


In [9]:
#sample 200 reviews
sampled_df = pd.DataFrame()
for star in range(1, 6):
    star_samples = df[df['stars'] == star].sample(n=40, random_state=42, replace=True) # taking exactly 40 samples for each star

    sampled_df = pd.concat([sampled_df, star_samples])

sampled_df = sampled_df.sample(n=200, random_state=42).reset_index(drop=True)

In [10]:
print(sampled_df['stars'].value_counts())

stars
3    40
1    40
4    40
2    40
5    40
Name: count, dtype: int64


In [11]:
for i in range(3):
    print(sampled_df.iloc[i]['stars'])
    print(sampled_df.iloc[i]['text'][:150])

3
I would give this place 3.5 stars.  The service was excellent and the view of Pheonix at sunset was nice.  I ate the wedge salad (nothing special) and
1
I don't like the pizza hear and if you go for lunch there are no specials by the slice.  The ranch tastes bottled and tastes gross.  The ingredients a
1
I wrote a less than stellar review about this place some time ago and decided that my prior awful experience might have been a fluke and as I love gre


## 2. Gemini Setup

In [12]:
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

In [13]:
model = genai.GenerativeModel('gemini-2.5-flash')

## 3. Prompt Designs

In [14]:
# Prompt A: Direct Classiication
PROMPT_A = """You are a Yelp review classifier. Given a restaurant review, predict the star rating (1 to 5 stars) where 1=worst, 5=best.

Return ONLY valid JSON with exactly these keys:
- "predicted_stars": integer between 1 and 5
- "explanation": brief explanation for your rating (1-2 sentences)

Review: "{review_text}"
"""

In [15]:
# Prompt B: Chain-of-Thought
PROMPT_B = """Analyze this Yelp review step by step:

1. Identify overall sentiment (positive/negative/neutral).
2. Note specific praises or complaints mentioned.
3. Determine intensity of sentiment (mild/strong).
4. Map to star rating: 1=very negative, 2=negative, 3=neutral/mixed, 4=positive, 5=very positive.

After analysis, output ONLY valid JSON with:
- "predicted_stars": integer 1-5
- "explanation": your reasoning based on steps above

Review: "{review_text}"
"""

In [16]:
# Prompt C: Multi-Aspect Scoring
PROMPT_C = """Evaluate this Yelp review on three aspects (score 1-5 each):
- Sentiment: How positive/negative is the language? (1=very negative, 5=very positive)
- Detail: How descriptive and substantive is the review? (1=vague, 5=detailed)
- Politeness: How respectful is the tone? (1=rude/aggressive, 5=polite/constructive)

Calculate average score and round to nearest integer for final stars (1-5).
Provide brief explanation based on the three scores.

Output ONLY valid JSON with:
- "predicted_stars": rounded average (integer 1-5)
- "explanation": mention each score briefly

Review: "{review_text}"
"""

In [17]:
prompts = {
    "Direct Classificaion": PROMPT_A,
    "Chain-of-Thought": PROMPT_B,
    "Multi-Aspect Scoring": PROMPT_C
}

## 4. Helper Functions

In [18]:
def extract_json_from_text(text):
    """Extract JSon from LLM respose"""
    try:
        start = text.find('{')
        end = text.rfind('}') + 1

        if start >= 0 and end > start:
            json_str = text[start:end]
            return json.loads(json_str)
        
    except json.JSONDecodeError:
        return None
    
@retry.Retry(timeout=30)
def call_gemini(prompt, max_retries=3):
    """Call gemini API"""
    for attempt in range(max_retries):
        try:
            response= model.generate_content(
                prompt,
                generation_config={'temperature':0.2}
            )
            return response.text.strip()
        
        except Exception as e:
            if '429' in str(e): # rate limit reached
                wait  = 2 ** attempt
                print(f"Rate limited. Waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"Error: {e}. Retry {attempt+1}/{max_retries}")
                time.sleep(1)

def evaluate_prompt(prompt_template, reviews_df, prompt_name):
    """Run classification for all reviews using a given prompt"""
    results = []
    valid_json_count = 0


    print(f"Evaluating: {prompt_name}...")

    for idx, row in reviews_df.iterrows():
        if idx % 20 == 0: # print details fpr 20 review batch to avoid clutter
            print(f"  Processed {idx}/{len(reviews_df)} reviews...")

        review_text = row['text']
        true_stars = row['stars']

        # prepare prompt
        prompt = prompt_template.format(review_text=review_text[:1000])

        # call LLM
        response_text = call_gemini(prompt)
        # print("RESPONSE TEXT>>>>>", response_text)

        parsed = extract_json_from_text(response_text)

        if parsed and 'predicted_stars' in parsed:
            predicted = int(parsed['predicted_stars'])
            explanation = parsed.get('explanation','')
            valid_json_count += 1
        else:
            predicted = None
            explanation= "JSON parsing failed"

        results.append({
            'true_stars': true_stars,
            'predicted_stars':predicted,
            'explanation':explanation,
            'response_raw':response_text,
            'json_valid':predicted is not None})
        
        time.sleep(1) # we avoid rate limits with a sight delay

    # create a resullts dataframe
    results_df = pd.DataFrame(results)

    # calculate metrics
    valid_results = results_df[results_df['json_valid']].copy()

    if len(valid_results) > 0:
        accuracy = accuracy_score(valid_results['true_stars'], valid_results['predicted_stars'])

        mae = mean_absolute_error(valid_results['true_stars'], valid_results['predicted_stars'])

    else:
        accuracy = 0
        mae = float('nan')


    return {
        'prompt_name': prompt_name,
        'resuts_df': results_df,
        'valid_json_count': valid_json_count,
        'valid_json_rate': valid_json_count / len(reviews_df),
        'accuracy':accuracy,
        'mae': mae,
        'sample_size':len(reviews_df)

    }

## 5. Run Experiments

In [19]:
all_results = {}

for prompt_name, prompt_template in prompts.items():
    results = evaluate_prompt(prompt_template, sampled_df, prompt_name)
    all_results[prompt_name]=results
    print(f"\n{prompt_name}:")
    print(f"  JSON Validity Rate: {results['valid_json_rate']:.2%}")
    print(f"  Accuracy: {results['accuracy']:.2%}")
    print(f"  MAE: {results['mae']:.3f}")


Evaluating: Direct Classificaion...
  Processed 0/200 reviews...
  Processed 20/200 reviews...
  Processed 40/200 reviews...
  Processed 60/200 reviews...
  Processed 80/200 reviews...
  Processed 100/200 reviews...
  Processed 120/200 reviews...
  Processed 140/200 reviews...
  Processed 160/200 reviews...
  Processed 180/200 reviews...

Direct Classificaion:
  JSON Validity Rate: 100.00%
  Accuracy: 59.00%
  MAE: 0.450
Evaluating: Chain-of-Thought...
  Processed 0/200 reviews...
  Processed 20/200 reviews...
  Processed 40/200 reviews...
  Processed 60/200 reviews...
  Processed 80/200 reviews...
  Processed 100/200 reviews...
  Processed 120/200 reviews...
  Processed 140/200 reviews...
  Processed 160/200 reviews...
  Processed 180/200 reviews...

Chain-of-Thought:
  JSON Validity Rate: 100.00%
  Accuracy: 57.50%
  MAE: 0.490
Evaluating: Multi-Aspect Scoring...
  Processed 0/200 reviews...
  Processed 20/200 reviews...
  Processed 40/200 reviews...
  Processed 60/200 reviews...
  P

## 6. Comparison

In [21]:
comparison_data = []
for prompt_name, results in all_results.items():
    comparison_data.append({
        'Prompt Strategy': prompt_name,
        'Sample Size': results['sample_size'],
        'JSON Validity Rate': f"{results['valid_json_rate']:.2%}",
        'Accuracy': f"{results['accuracy']:.2%}",
        'MAE': f"{results['mae']:.3f}",
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

     Prompt Strategy  Sample Size JSON Validity Rate Accuracy   MAE
Direct Classificaion          200            100.00%   59.00% 0.450
    Chain-of-Thought          200            100.00%   57.50% 0.490
Multi-Aspect Scoring          200            100.00%   33.50% 0.820


## 7. Discussion

### Performance Summary
The evaluation reveals surprising patterns in prompt effectiveness for Yelp review classification. Contrary to expectations, the simplest approach (Direct Classification) outperformed more structured methods, achieving the highest accuracy at 59.00% with the lowest Mean Absolute Error (0.450). Chain-of-Thought prompting followed closely at 57.50% accuracy, while Multi-Aspect Scoring significantly underperformed at only 33.50% accuracy with substantially higher error (MAE: 0.820).

### Key Findings

**1. Simplicity Outperforms Complexity**
Direct Classification's superior performance suggests that for star rating prediction, straightforward instructions may be more effective than elaborate reasoning frameworks. The LLM appears capable of intuitive rating assessment without explicit step-by-step guidance.

**2. JSON Validity Excellence**
All three approaches achieved perfect 100% JSON validity rates, demonstrating that explicit formatting instructions combined with robust parsing effectively ensure consistent, machine-readable outputs.

**3. Multi-Aspect Scoring Limitations**
The poor performance of Multi-Aspect Scoring (33.50% accuracy) indicates significant issues with this approach. The three-dimensional scoring system (sentiment, detail, politeness) appears to introduce unnecessary complexity and potential scoring contradictions that degrade final rating accuracy.

**4. Error Patterns**
The MAE values reveal distinct error characteristics:
- Direct Classification: Most accurate with smallest average error (0.450 stars off)
- Chain-of-Thought: Slightly higher but comparable error (0.490)
- Multi-Aspect Scoring: Substantially higher error (0.820), indicating frequent misclassifications

### Limitations and Observations

1. **Accuracy Ceiling**: All methods performed below 60% accuracy, suggesting inherent challenges in 5-class rating prediction from text alone, possibly due to:
   - Subjectivity in human ratings
   - Sarcasm or nuanced language
   - Contextual factors beyond review text

2. **Dataset Characteristics**: The Yelp dataset contains diverse writing styles and varying review lengths that may challenge consistent classification.

3. **LLM Capabilities**: Gemini's performance indicates capability for the task but suggests potential benefits from:
   - Few-shot examples
   - Domain-specific fine-tuning
   - Ensemble approaches combining multiple strategies

### Conclusion
Prompt engineering significantly impacts classification performance, but simpler approaches can outperform more complex ones. The optimal strategy balances accuracy, reliability, and computational efficiency. For Yelp review rating prediction, Direct Classification provides the best balance of these factors, achieving reasonable accuracy with perfect output reliability.